## Étape 1 : découpage du fichier de questions-réponses

Consigne du prof : garder 100 questions pour l'évaluation et utiliser le reste dans le système.

Ce notebook crée deux tables dans `PROJECT_DB.PUBLIC` :
- `EVAL_TICKETS` : 100 questions réservées à l'évaluation, que le système ne doit jamais consulter ;
- `KB_QA` : les 493 autres, utilisables comme base de connaissances.

Le tirage est fait une seule fois, avec une graine fixe, en respectant les proportions de `Theme` et de `Status`.

Le fichier Excel est lu directement dans le stage `@PROJECT_DB.PUBLIC.FILES`.

In [ ]:
!pip install openpyxl

### 1. Lecture du fichier dans le stage

La cellule liste les fichiers Excel du stage, repère celui des questions-réponses (son nom contient `aqua_spot`) et le charge.

In [ ]:
import io

import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

STAGE = "@PROJECT_DB.PUBLIC.FILES"
SEED = 42        # graine fixe : ne pas la changer une fois le tirage fait
N_EVAL = 100

rows = session.sql(
    f"SELECT RELATIVE_PATH FROM DIRECTORY({STAGE}) WHERE RELATIVE_PATH ILIKE '%.xlsx'"
).collect()
xlsx_files = [r["RELATIVE_PATH"] for r in rows]
print("Fichiers Excel du stage :")
for f in xlsx_files:
    print(" -", f)

matches = [f for f in xlsx_files if "aqua_spot" in f.lower()]
assert len(matches) == 1, f"Fichier de questions-réponses introuvable ou ambigu : {matches}"
FILE = matches[0]
print("\nFichier retenu :", FILE)

with session.file.get_stream(f"{STAGE}/{FILE}") as stream:
    raw = pd.read_excel(io.BytesIO(stream.read()))
print(raw.shape)

assert raw["ID"].is_unique, "Des identifiants sont en double"
assert len(raw) > N_EVAL
raw.head(3)

### 2. Nettoyage léger

- la colonne `Team` contient le nom de l'équipe et son adresse dans la même cellule : on les sépare ;
- la colonne `Scope` est entièrement vide : on la retire ;
- les noms de colonnes passent en majuscules, comme le veut Snowflake.

In [ ]:
df = raw.copy()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

parts = df["Team"].str.split("\n", n=1, expand=True)
df["Team_name"] = parts[0].str.strip()
df["Team_email"] = parts[1].str.strip() if parts.shape[1] > 1 else None

if df["Scope"].isna().all():
    df = df.drop(columns=["Scope"])

df.columns = [c.upper() for c in df.columns]
df = df.sort_values("ID").reset_index(drop=True)

print(df.columns.tolist())
df[["ID", "THEME", "STATUS", "TEAM_NAME", "TEAM_EMAIL"]].head()

### 3. Tirage stratifié

Chaque groupe (thème, statut) reçoit un nombre de questions proportionnel à sa taille, arrondi de façon à tomber exactement sur 100.

In [ ]:
def stratified_ids(data, by, n, seed):
    sizes = data.groupby(by).size()
    quota = sizes * n / len(data)
    take = quota.astype(int)
    # on distribue les places restantes aux plus grands restes
    rest = (quota - take).sort_values(ascending=False, kind="stable")
    for key in rest.index[: n - int(take.sum())]:
        take[key] += 1
    ids = []
    for key, group in data.groupby(by):
        if take[key] > 0:
            ids += group.sample(n=int(take[key]), random_state=seed)["ID"].tolist()
    return sorted(ids)


eval_ids = stratified_ids(df, ["THEME", "STATUS"], N_EVAL, SEED)

eval_df = df[df["ID"].isin(eval_ids)].reset_index(drop=True)
kb_df = df[~df["ID"].isin(eval_ids)].reset_index(drop=True)

assert len(eval_df) == N_EVAL
assert len(eval_df) + len(kb_df) == len(df)
assert set(eval_df["ID"]).isdisjoint(set(kb_df["ID"]))

print("Évaluation :", len(eval_df), "| Base de connaissances :", len(kb_df))

### 4. Contrôle des proportions

Les deux colonnes en pourcentage doivent être proches.

In [ ]:
def compare(col):
    return pd.DataFrame({
        "total": df[col].value_counts(),
        "eval": eval_df[col].value_counts(),
        "% total": (df[col].value_counts(normalize=True) * 100).round(1),
        "% eval": (eval_df[col].value_counts(normalize=True) * 100).round(1),
    }).fillna(0)


print(compare("STATUS"), "\n")
print(compare("THEME"), "\n")
print(compare("FREQUENCY"))

### 5. Écriture des deux tables dans Snowflake

In [ ]:
for name, data in (("EVAL_TICKETS", eval_df), ("KB_QA", kb_df)):
    session.write_pandas(
        data, name,
        database="PROJECT_DB", schema="PUBLIC",
        auto_create_table=True, overwrite=True,
    )
    print(name, "écrite :", len(data), "lignes")

### 6. Vérification dans Snowflake

Attendu : 100 et 493 lignes, et aucun identifiant commun.

In [ ]:
check = session.sql("""
    SELECT
        (SELECT COUNT(*) FROM PROJECT_DB.PUBLIC.EVAL_TICKETS) AS NB_EVAL,
        (SELECT COUNT(*) FROM PROJECT_DB.PUBLIC.KB_QA) AS NB_KB,
        (SELECT COUNT(*) FROM PROJECT_DB.PUBLIC.EVAL_TICKETS e
           JOIN PROJECT_DB.PUBLIC.KB_QA k ON e.ID = k.ID) AS NB_COMMUNS
""").collect()[0]
print(check)